# Fault-labelled ML training data

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mosesfalowo/ogbonlab-notebooks/blob/main/notebooks/ssm_fault_ml_dataset.ipynb)


Generated by [OgbonLab](https://ogbonlab.com), Synthetic Seismic Modeling course.

**Assumptions**: Course defaults. Edit the parameters cell and run again; every value is yours to change.

**Workflow**:
1. 3.5 Fault-Detection Training Pairs
2. 10.1 Fault-Detection Dataset
3. 9.5 Export Your Model

In [ ]:
# Ask Ogbon starter: Generate ML training data
# Generated by OgbonLab (ogbonlab.com), Synthetic Seismic Modeling course.
"""Fault-labelled training pairs (section, label) by 2D convolution.

Follows Sections 3.4 (Convolving a Section), 3.5 (Fault-Detection Training
Pairs) and 10.1 (Fault-Detection Dataset); Section 9.5 exports the result.
Units: samples of 1 ms, impedance m/s times g/cc, memory MiB of float32.
Assumptions and limits:
- Each trace is convolved on its own (vertical incidence), so a fault is a
  sharp step; a wave equation draws a ramp about a Fresnel zone wide and adds
  diffractions (Sections 3.6 and 10.1).
- The figure's pair: one vertical fault of one throw in flat beds, no noise; the
  counts are for N_SAMPLES of it.
- The field: N_BUILD members of the field earth's family, each from its own seed:
  fold, dip, one to three faults and their throw, an unconformity or none, the
  wavelet's peak, Q and the signal-to-noise vary. The label is read from the same
  cells as the section, so it is exact; they are saved to OUT_FILE.

It prints the numbers Figure 12.5 computes for the same answers as name = value
lines; then it builds the field earth every starter shares, prints what it measures
there as [field] name = value lines, and draws one figure. Python 3.9+, numpy and
matplotlib; the earth's random numbers are seeded, so every run prints the same.
Edit the parameters and run again.
"""
import numpy as np
import matplotlib.pyplot as plt

# --- parameters from your interview ---
N_SAMPLES = 2000   # training sections in the full dataset
GRID = 128         # pixels per side: GRID traces of GRID samples (1 ms)
LABEL = 'fault'    # 'fault', 'horizon' or 'facies'
BUDGET = 'laptop'  # the compute you said you have
F_HZ = 30.0        # Ricker peak frequency (Hz), the book's standard

In [ ]:
# --- the book's constants (not interview answers) ---
# the book's standard rocks (SSM.STANDARD, Section 3.3): P velocity (m/s), S velocity (m/s), density (g/cc)
ROCKS = {
    'shale': (2800, 1241.3793103448277, 2.2521031547808517),
    'brine sand': (2889.973997793982, 1504.4719169684909, 2.2545487499999997),
    'oil sand': (2543.1993596875604, 1529.5017037311773, 2.1813626422963988),
    'gas sand': (2341.92404704112, 1582.4208998550723, 2.0379042775100316)
}
BEDS = ['shale', 'brine sand', 'shale', 'gas sand', 'shale', 'oil sand']   # cycled downward
BED_SAMPLES = 8         # a bed is 8 ms of two-way time
THROW = 6               # the fault's throw (samples); the right-hand block drops
BYTES_PER_FLOAT32 = 4
MIB = 1048576.0         # bytes in a mebibyte
N_BUILD, SEED0 = 32, 100   # field sections built and saved here (set N_BUILD = N_SAMPLES for the full set), first seed
WINDOW_MS, T_MAX_MS = 512.0, 1100.0   # every field section spans 512 ms, cut from the first 1100 ms
OUT_FILE = 'field_training_pairs.npz'

def report(name, value, fmt='%.6f'):
    """Print one result as name = value, the line Figure 12.5 checks against the book's kernel."""
    print(('%s = ' + fmt) % (name, value))

def style(ax, title, xlabel, ylabel, grid=True):
    """The starters' one look: a title, both axes labelled with their units, a light grid."""
    ax.set_title(title, fontsize=10)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if grid:
        ax.grid(True, alpha=0.3)

MIN_HALF_MS = 40.0      # the book's tail rule (Figure 0.1): never shorter than 40 ms either side
TAIL_MS_HZ = 1200.0     # ... and 1.20 / f seconds, 1200 / f ms, either side

def ricker(f_hz, dt_ms, n):
    """The zero-phase Ricker of peak frequency f_hz (Hz) on n samples of dt_ms (ms), peak at the centre (Section 1.1)."""
    t = (np.arange(n) - n // 2) * dt_ms * 1e-3       # s
    a = (np.pi * f_hz * t) ** 2
    return (1 - 2 * a) * np.exp(-a)

def ricker_full(f_hz, dt_ms=1.0):
    """The Ricker taken whole by the book's tail rule: 81 taps at 30 Hz on 1 ms."""
    half = max(int(round(MIN_HALF_MS / dt_ms)), int(np.ceil(TAIL_MS_HZ / (f_hz * dt_ms))))
    return ricker(f_hz, dt_ms, 2 * half + 1)

# --- the Ogbon field earth: one seeded 2D earth that every starter shares ---
# In geological order: thin sand and shale beds with a regional dip, folded into an
# asymmetric anticline and cut by normal faults, the largest a listric growth fault
# (its throw grows with depth and its hanging wall rolls over into it); uplift, and an
# angular unconformity that truncates the crest; the reservoir sand subcrops beneath
# it, wedging out against it, shaling out to the left, sealed by a shale cap and
# holding gas above a flat contact; then a draped overburden with a channel. Each
# cell's velocity follows the
# compaction trend at its deepest burial (the rocks under the unconformity were once
# buried deeper), sands soft when shallow and hard when deep; Gardner's density; the
# mudrock and sandstone lines for S. Units: m, m/s, g/cc; times in ms.
FAULTS = [(0.66, 1.0, 1.4, 380.0), (0.80, -1.0, 0.6, 470.0), (0.22, -1.0, 0.5, 560.0)]   # (x / length, side, size, tip m)
COMPACTION = (1900.0, 1.4)                             # P velocity = 1900 m/s + 1.4 per metre of burial, before lithology
K_RES = 9                                              # the reservoir: bed 9 of the deep package, 70 m thick
SEAL_M = 30.0                                          # the shale cap on the reservoir's top: 30 m, and never under two cells

def field_earth(nx=240, nz=320, dx=15.0, dz=5.0, seed=7, crest=0.45, fold=1.0, dip=1.0, faults=None, throw=1.0,
                unconformity=True, seal=None, upper=None, lower=None, column_m=130.0):
    """The field earth on nz by nx cells of dz by dx (m). seal, upper and lower are (vp, vs, density) of the shale on the
    unconformity and of the reservoir above and below its flat contact, or None for the compaction trend's own rocks."""
    rng = np.random.default_rng(seed)
    x, z = np.arange(nx) * dx, np.arange(nz) * dz
    xn, zz, sc = x / x[-1], z[:, None] + 0.0 * x, nz * dz / 1600.0     # sc: the geometry scales with the grid's depth
    hump = np.exp(-0.5 * ((xn - crest) / np.where(xn < crest, 0.13, 0.25)) ** 2)   # the anticline, steep to the left
    swell = 1.0 + 0.15 * np.sin(2 * np.pi * (0.8 * xn + 0.37 * np.arange(26)[:, None]))   # each bed thickens and thins
    thick = sc * (24.0 + 22.0 * rng.random(26))[:, None] * swell
    thick[K_RES] = sc * 70.0
    beds = sc * 470.0 + np.cumsum(thick, axis=0) - thick + sc * (170.0 * dip * xn - 300.0 * fold * hump)
    if faults is not None:
        fs = [(rng.uniform(0.12, 0.88), rng.choice([-1.0, 1.0]), rng.uniform(0.4, 1.4), rng.uniform(300.0, 600.0))
              for k in range(faults)]
    fault = np.zeros((nz, nx), bool)
    for x0, side, size, tip in (FAULTS if faults is None else fs):
        tip, a = tip * sc, 0.2 / (1200.0 * sc) ** 1.3       # the plane: x / length = x0 + side a (depth - tip) to the 1.3, listric
        plane = np.where(side * (xn - x0) > 0, tip + (np.abs(xn - x0) / a) ** (1 / 1.3), -np.inf)   # its depth at each x
        for k in range(26):
            g = throw * size * sc * 90.0 * np.clip((beds[k] - tip) / (700.0 * sc), 0.0, 1.6)      # growth: deeper beds slip more
            xk = x0 + side * a * np.maximum(beds[k] - tip, 0.0) ** 1.3                          # where the plane cuts the bed
            g = g * (1.0 + 0.6 * np.exp(-np.abs(xn - xk) / 0.04))                                # the rollover into the fault
            beds[k] = beds[k] + np.clip(plane - beds[k], 0.0, g)       # the hanging wall drops; in the heave the bed lies on the plane
        ix = np.rint((x0 + side * a * np.maximum(z - tip, 0.0) ** 1.3) * (nx - 1)).astype(int)
        on = (z > tip) & (ix >= 0) & (ix < nx)
        fault[np.nonzero(on)[0], ix[on]] = True
    beds = np.maximum.accumulate(beds, axis=0)
    if unconformity:                                   # cut just under the reservoir's crest, so it subcrops on both flanks
        unc = beds[K_RES + 1, np.argmax(hump)] + sc * (15.0 + 160.0 * dip * (xn - crest)
                                                       + 10.0 * np.sin(2 * np.pi * (1.3 * xn + 0.2)))
    else:
        unc = beds[0] - sc * 30.0                      # no erosion: the overburden lies on the beds
    fault &= zz > unc                                  # the faults are older than the unconformity
    fr = np.cumsum(0.6 + rng.random(13))
    fr = (fr - fr[0]) / (fr[-1] - fr[0])
    over = [sc * 40.0 + (unc.min() - sc * 90.0) * f + 0.85 * f * f * (unc - unc.min()) - sc * 25.0 * f * hump for f in fr]
    tops = np.maximum.accumulate(np.vstack(over + list(np.maximum(beds, unc))), axis=0)
    lay = (zz[:, None, :] >= tops[None, :, :]).sum(axis=1)                    # layer 13 is the seal; bed k is layer 14 + k
    sand = np.where(lay < 13, rng.random(41)[lay] < 0.45, (lay - 14) % 2 == 1) & (lay > 0) & (lay != 13)
    chan = (zz >= over[5]) & (zz < over[5] + sc * 45.0 * np.exp(-0.5 * ((xn - 0.7) / 0.04) ** 2))
    res = lay == 14 + K_RES
    top = np.where(res, zz, np.inf).min(axis=0)
    cap = (zz < top) & (zz >= top - max(SEAL_M * sc, 2.0 * dz))   # the seal: shale on the sand's top, under a bed or the unconformity
    sand = np.where(cap, False, sand)
    burial = zz + np.where(lay >= 14, sc * 150.0 * hump, 0.0)                # uplift: the deep rocks were buried deeper
    vary = 1.0 + 0.035 * np.clip(rng.standard_normal(41), -1.2, 1.2)        # bed to bed variety
    vp = (COMPACTION[0] + COMPACTION[1] * burial) * vary[lay]
    vp = np.where(sand | chan, vp * (0.86 + 0.1 * np.minimum(burial / 1000.0, 1.8)), vp)
    vp = np.where(chan, vp * 0.92, vp)                 # the channel cuts a horizon and is filled with a soft sand
    sand |= chan
    vs = np.where(sand, 0.804 * vp - 856.0, (vp - 1360.0) / 1.16)            # the sandstone and mudrock lines (Castagna)
    den = 1.741 * (vp / 1000.0) ** 0.25                                      # Gardner
    ntg = np.clip((xn - 0.12) / 0.12, 0.0, 1.0)                              # the reservoir shales out to the left
    contact = float(top.min()) + column_m * sc         # the flat contact, column_m under the shallowest sand
    for rock, on in ((seal, (lay == 13) | cap), (upper, res & (zz < contact)), (lower, res & (zz >= contact))):
        if rock is not None:
            w = 1.0 if rock is seal else ntg
            vp, vs, den = [np.where(on, w * rock[i] + (1 - w) * q, q) for i, q in enumerate((vp, vs, den))]
    return dict(x=x, z=z, vp=vp, vs=vs, den=den, lay=lay, sand=sand, res=res, fault=fault, unc=unc, top=top,
                contact=contact, ntg=ntg, dx=dx, dz=dz, chan=chan, cap=cap, gas=res & (zz < contact) & (ntg > 0.5))

def field_time(m, dt_ms, t_max_ms):
    """Each trace from depth to two-way time by vertical rays: every cell array of m on a dt_ms grid to t_max_ms."""
    t_top = 2000.0 * np.cumsum(m['dz'] / m['vp'], axis=0) - 2000.0 * m['dz'] / m['vp']
    t = np.arange(0.0, t_max_ms, dt_ms)
    pick = np.array([np.searchsorted(t_top[:, i], t, 'right') - 1 for i in range(t_top.shape[1])]).T
    cols = np.arange(t_top.shape[1])[None, :]
    out = dict((k, m[k][pick, cols]) for k in ('vp', 'vs', 'den', 'lay', 'sand', 'res', 'gas'))
    seen = np.vstack([0 * cols, np.cumsum(m['fault'], axis=0)])                 # fault cells above each row
    after = np.maximum(np.vstack([pick[1:], pick[-1:] + 1]), pick + 1)
    out['fault'] = seen[np.minimum(after, seen.shape[0] - 1), cols] > seen[pick, cols]   # a fault cell inside the sample
    out['t'], out['t_top'] = t, t_top
    return out

def reflectivity(ai):
    """Normal-incidence coefficient at the top of every sample, down axis 0 (Section 2.1)."""
    rc = np.zeros_like(ai)
    rc[1:] = (ai[1:] - ai[:-1]) / (ai[1:] + ai[:-1])
    return rc

def q_peak_hz(f0, t_s, q):
    """Peak (Hz) of a Ricker of peak f0 after t_s seconds through constant Q: the root of 2 f f / (f0 f0) + (pi t / Q) f = 2."""
    b = np.pi * t_s / q
    return (-b + np.sqrt(b * b + 16.0 / f0 ** 2)) * f0 ** 2 / 4.0

def field_image(rc, dt_ms, f0, q=80.0, snr=None, seed=11):
    """Every trace convolved with a zero-phase Ricker whose peak falls with time as constant q says, plus coloured noise."""
    nt, dt = rc.shape[0], dt_ms * 1e-3
    f = np.fft.rfftfreq(2 * nt, dt)[:, None]
    def band(fp):                                      # the Ricker's spectrum (Section 1.1), over dt for a sum of samples
        return 2.0 / np.sqrt(np.pi) * f ** 2 / fp ** 3 * np.exp(-(f / fp) ** 2) / dt
    def filt(a, fp):
        return np.fft.irfft(np.fft.rfft(a, 2 * nt, axis=0) * band(fp), 2 * nt, axis=0)[:nt]
    fp = q_peak_hz(f0, np.arange(nt) * dt, q)
    knots = np.linspace(fp.min(), fp.max(), 4 if fp.max() - fp.min() > 1e-9 else 1)
    step = max((fp.max() - fp.min()) / 3.0, 1e-9)
    out = sum(np.clip(1.0 - np.abs(fp - k) / step, 0.0, 1.0)[:, None] * filt(rc, k) for k in knots)
    if snr:
        noise = filt(np.random.default_rng(seed).standard_normal(rc.shape), fp.mean())
        noise = (noise + np.roll(noise, 1, 1) + np.roll(noise, -1, 1)) / 3.0          # coherent over a few traces
        out = out + noise * np.sqrt((out ** 2).mean() / (noise ** 2).mean()) / snr
    return out

def fieldnote(name, value, fmt='%.3f'):
    """Print a number of the field earth as [field] name = value: computed here, not by Figure 12.5."""
    print(('[field] %s = ' + fmt) % (name, value))

def show(ax, a, x_m, y, title, ylabel='two-way time (ms)', clip=None):
    """A section as an image, black for positive, clipped at clip (or at its 99th percentile): km across, time or depth down."""
    c = clip if clip else np.percentile(np.abs(a), 99.0) + 1e-12
    ax.imshow(a, cmap='gray_r', vmin=-c, vmax=c, aspect='auto', extent=[x_m[0] / 1e3, x_m[-1] / 1e3, y[-1], y[0]])
    style(ax, title, 'distance (km)', ylabel, grid=False)

def validate(n_samples, grid, label, throw):
    """Stop with a clear message on a parameter the model cannot use."""
    if label not in ('fault', 'horizon', 'facies'):
        raise ValueError("LABEL must be 'fault', 'horizon' or 'facies', not %r" % (label,))
    if n_samples < 1 or grid < 4 or not 0 <= throw < grid:
        raise ValueError('need N_SAMPLES >= 1, GRID >= 4 and a throw smaller than the section')

def column(n, drop, rocks):
    """Impedance (m/s x g/cc) and bed index of one trace of n samples, the beds moved down by drop."""
    idx = np.maximum(np.arange(n) - drop, 0) // BED_SAMPLES
    names = [BEDS[i % len(BEDS)] for i in idx]
    return np.array([rocks[m][0] * rocks[m][2] for m in names]), idx

def training_pair(n, throw, w, label, rocks):
    """One n-by-n section, trace by trace (Section 3.4), and its label; the fault plane at trace n // 2."""
    X = np.zeros((n, n)); facies = np.zeros((n, n)); horizon = np.zeros((n, n)); fault = np.zeros((n, n))
    for ix in range(n):
        z, idx = column(n, throw if ix >= n // 2 else 0, rocks)
        rc = np.zeros(n); rc[1:] = (z[1:] - z[:-1]) / (z[1:] + z[:-1])
        X[:, ix] = np.convolve(rc, w)[w.size // 2:w.size // 2 + n]
        horizon[:, ix] = (rc != 0)
        facies[:, ix] = idx % len(BEDS)
    fault[:, n // 2 - 1] = 1; fault[:, n // 2] = 1     # one cell each side of the plane
    return X, {'fault': fault, 'horizon': horizon, 'facies': facies}[label]

def realisation(seed, n, label):
    """One member of the field earth's family on n traces of n samples, and its label, pixel for pixel."""
    rng = np.random.default_rng(seed)
    knobs = dict(crest=rng.uniform(0.25, 0.75), fold=rng.uniform(0.2, 1.5), dip=rng.uniform(-1.0, 1.5),
                 faults=int(rng.integers(1, 4)), throw=rng.uniform(1.0, 2.5), unconformity=bool(rng.random() < 0.6))
    e = field_earth(nx=n, dx=3600.0 / n, seed=seed, **knobs)
    dt = WINDOW_MS / n                                         # every section spans the same window
    t = field_time(e, dt, T_MAX_MS)
    k0 = int(rng.uniform(300.0, T_MAX_MS - WINDOW_MS) / dt)
    snr = rng.uniform(2.0, 10.0)
    img = field_image(reflectivity(t['vp'] * t['den']), dt, rng.uniform(20.0, 40.0), q=rng.uniform(50.0, 150.0), snr=snr, seed=seed)
    lab = {'fault': t['fault'], 'horizon': np.vstack([0 * t['lay'][:1], np.diff(t['lay'], axis=0)]) != 0,
           'facies': t['sand'].astype(int) + t['res']}[label]
    knobs['snr'] = snr
    return img[k0:k0 + n].astype(np.float32), lab[k0:k0 + n].astype(np.uint8), knobs

def plot_family(xs, ys, notes, label):
    """Four sections of the family and their labels."""
    fig, ax = plt.subplots(2, 4, figsize=(13, 6.2))
    for k in range(min(4, len(xs))):
        c = np.percentile(np.abs(xs[k]), 99)
        ax[0, k].imshow(xs[k], cmap='gray_r', vmin=-c, vmax=c, aspect='auto')
        style(ax[0, k], notes[k], 'trace', 'time sample' if k == 0 else '', grid=False)
        ax[1, k].imshow(ys[k], cmap='magma' if label == 'fault' else 'viridis', aspect='auto', interpolation='nearest')
        style(ax[1, k], 'label: ' + label, 'trace', 'time sample' if k == 0 else '', grid=False)
    fig.suptitle('Training pairs from the field earth\'s family: fold, dip, faults, throw, unconformity, wavelet, Q and noise vary', fontsize=10)
    fig.tight_layout()
    plt.show()

def main():
    validate(N_SAMPLES, GRID, LABEL, THROW)
    w = ricker_full(F_HZ)
    X, y = training_pair(GRID, THROW, w, LABEL, ROCKS)
    taps = w.size
    report('taps', taps, '%d')
    report('multiply_adds_per_section', GRID * GRID * taps, '%d')       # traces x samples x taps (Figure 0.2)
    report('multiply_adds_dataset', N_SAMPLES * GRID * GRID * taps, '%d')
    report('dataset_mib', N_SAMPLES * GRID * GRID * BYTES_PER_FLOAT32 / MIB, '%.3f')
    report('section_rms', np.sqrt((X ** 2).mean()))
    report('label_cells', np.count_nonzero(y), '%d')
    report('label_classes', np.unique(y).size, '%d')
    # self-check: the label is pixel for pixel the section's shape
    assert X.shape == y.shape == (GRID, GRID)
    # the field: N_BUILD members of the field earth's family, each with its own seed, saved for training
    n = min(N_BUILD, N_SAMPLES)
    pairs = [realisation(SEED0 + k, GRID, LABEL) for k in range(n)]
    xs = np.stack([p[0] for p in pairs]); ys = np.stack([p[1] for p in pairs])
    assert xs.shape == ys.shape == (n, GRID, GRID)
    np.savez_compressed(OUT_FILE, sections=xs, labels=ys, seeds=SEED0 + np.arange(n),
                        faults=[p[2]['faults'] for p in pairs], snr=[p[2]['snr'] for p in pairs])
    fieldnote('sections_built', n, '%d')
    fieldnote('label_fraction', ys.astype(bool).mean(), '%.4f')
    fieldnote('faults_per_section', np.mean([p[2]['faults'] for p in pairs]), '%.2f')
    fieldnote('with_unconformity', np.mean([p[2]['unconformity'] for p in pairs]), '%.2f')
    fieldnote('saved_mib', xs.nbytes / MIB + ys.nbytes / MIB, '%.3f')
    print('[field] saved %s: sections %s float32, labels %s uint8' % (OUT_FILE, xs.shape, ys.shape))
    notes = ['seed %d: %d fault%s, %s, S/N %.0f' % (SEED0 + k, p[2]['faults'], 's' if p[2]['faults'] > 1 else '',
             'unconformity' if p[2]['unconformity'] else 'conformable', p[2]['snr']) for k, p in enumerate(pairs[:4])]
    plot_family(xs[:4], ys[:4], notes, LABEL)

if __name__ == '__main__':
    main()

## Where to go next

- The course: https://ogbonlab.com/textbooks/synthetic-seismic-modeling
- More notebooks: https://github.com/mosesfalowo/ogbonlab-notebooks
- Scale up: Devito, SPECFEM3D, Madagascar (course Part 12.3)